In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate PyMuPDF python-docx

import os
import faiss
import numpy as np
import fitz
from docx import Document
from google.colab import files

print("Upload your Sanskrit files (.txt / .pdf / .docx)")
uploaded = files.upload()

# =========================
# LOADER
# =========================
def load_documents(uploaded_files):
    texts = []

    for filename in uploaded_files:
        try:
            content = ""

            if filename.endswith(".txt"):
                with open(filename, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()

            elif filename.endswith(".pdf"):
                doc = fitz.open(filename)
                for page in doc:
                    content += page.get_text()

            elif filename.endswith(".docx"):
                doc = Document(filename)
                for para in doc.paragraphs:
                    content += para.text + "\n"

            else:
                print(f"❌ Unsupported file skipped: {filename}")
                continue

            if content.strip():
                texts.append(content)
                print(f"✅ Loaded: {filename}")
            else:
                print(f"⚠ Empty content: {filename}")

        except Exception as e:
            print(f"❌ Error reading {filename}: {e}")

    return texts

documents = load_documents(uploaded)

print(f"\nTotal documents: {len(documents)}")

if len(documents) == 0:
    raise ValueError("No valid documents loaded — CHECK FILE FORMAT")

In [ ]:
from sentence_transformers import SentenceTransformer

# =========================
# SANSKRIT CHUNKER
# =========================
def sanskrit_chunker(text, chunk_size=300):
    text = text.replace("॥", "।")
    sentences = [s.strip() for s in text.split("।") if s.strip()]

    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) < chunk_size:
            current += sent + "।"
        else:
            chunks.append(current.strip())
            current = sent + "।"

    if current:
        chunks.append(current.strip())

    return chunks

# =========================
# CREATE CHUNKS
# =========================
all_chunks = []
for doc in documents:
    all_chunks.extend(sanskrit_chunker(doc))

print("Total chunks:", len(all_chunks))

if len(all_chunks) == 0:
    raise ValueError("Chunking failed — document may be empty")

print("\nSample chunk:\n", all_chunks[0][:200])

# =========================
# EMBEDDING MODEL
# =========================
embedder = SentenceTransformer("intfloat/multilingual-e5-large")

# =========================
# EMBEDDINGS
# =========================
embeddings = embedder.encode(
    ["passage: " + t for t in all_chunks],
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings)

if len(embeddings.shape) == 1:
    embeddings = embeddings.reshape(1, -1)

print("Embedding shape:", embeddings.shape)

# =========================
# FAISS
# =========================
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("✅ FAISS ready")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✅ mT5 loaded (fixed mode)")

# =========================
# RETRIEVE
# =========================
def retrieve(query, k=3):
    q_emb = embedder.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(q_emb, k)
    return [all_chunks[i] for i in indices[0]]


def build_prompt(query, context):
    return f"""
Extract the exact answer from the Sanskrit context.

Return ONLY a relevant sentence from the context.

Context:
{context}

Question:
{query}

Answer in Sanskrit:
"""


def extract_from_chunks(query, chunks):
    # simple keyword fallback if model fails
    for chunk in chunks:
        if any(word in chunk for word in query.split()):
            return chunk[:200]
    return "उत्तर उपलब्ध नहीं है"

# =========================
# QA FUNCTION
# =========================
def ask(query):
    chunks = retrieve(query)

    print("\n--- RETRIEVED ---")
    for i, c in enumerate(chunks):
        print(f"\nChunk {i+1}:\n{c}")

    context = "\n".join(chunks)
    prompt = build_prompt(query, context)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if "<extra_id_" in answer or len(answer) < 5:
        answer = extract_from_chunks(query, chunks)

    print("\n--- ANSWER ---")
    print(answer)

# =========================
# LOOP
# =========================
while True:
    q = input("\nAsk (or exit): ")
    if q.lower() == "exit":
        break
    ask(q)

In [ ]:
def split_sentences(text):
    text = text.replace("॥", "।")
    sentences = [s.strip() + "।" for s in text.split("।") if s.strip()]
    return sentences


def smart_answer(query, chunks):
    query_emb = embedder.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    best_score = -1
    best_sentence = ""

    for chunk in chunks:
        sentences = split_sentences(chunk)

        sent_embs = embedder.encode(
            ["passage: " + s for s in sentences],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        scores = np.dot(sent_embs, query_emb)

        max_idx = np.argmax(scores)

        if scores[max_idx] > best_score:
            best_score = scores[max_idx]
            best_sentence = sentences[max_idx]

    return best_sentence if best_sentence else "उत्तर उपलब्ध नहीं है"


def ask_improved(query):
    chunks = retrieve(query)

    print("\n--- RETRIEVED ---")
    for i, c in enumerate(chunks):
        print(f"\nChunk {i+1}:\n{c}")

    answer = smart_answer(query, chunks)

    print("\n--- IMPROVED ANSWER ---")
    print(answer)

while True:
    q = input("\nAsk improved (or exit): ")
    if q.lower() == "exit":
        break
    ask_improved(q)

In [ ]:
test_queries = [
    "श्वानशावकस्य किं अभवत् ?",
    "शंखनादः कुत्र गच्छति ?",
    "वानराः किं कुर्वन्ति ?"
]

print("\n===== DEMO OUTPUT =====\n")

for q in test_queries:
    print(f"\n🔹 Question: {q}")

    chunks = retrieve(q)

    # show top chunk only (clean demo)
    print("\nTop Context:")
    print(chunks[0][:200], "...")

    # use improved smart answer
    answer = smart_answer(q, chunks)

    print("\nAnswer:")
    print(answer)

    print("\n" + "="*50)